### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are paisings of:

    1. A schema including the name of the tool, a description, add/or argument definitions(often a JSON schema)
    2. A function or coroutine to execute.

In [53]:
from langchain.tools import tool as langchain_tool
from langchain_ollama import ChatOllama
import json

@langchain_tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

model_with_tool = model.bind_tools([get_weather])


def structure_model_response(result):
    messages_data = [
        message.model_dump()
        for message in result["messages"]
    ]

    print(json.dumps(messages_data, indent=2, default=str))
    

In [43]:
response = model_with_tool.invoke(
    "What's the weather like in Boston?"
)
print('contents',response.content)
print('tools', response.tool_calls)
for tool_call in response.tool_calls:
    print('tool_name',tool_call['name'])
    print('tool_arg', tool_call['args']['location'])
    if tool_call['name'] == 'get_weather':
       result = get_weather.invoke(tool_call["args"])
       print("Tool result:", result)
    

contents 
tools [{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'd7dacf7b-1ea0-45f0-b89d-db09e4f9b2b1', 'type': 'tool_call'}]
tool_name get_weather
tool_arg Boston
Tool result: It's sunny in Boston


### Get response message from LLM on given question

In [44]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Human's question
question = "What's the weather like in Boston?"

# 2. Ask the model to decide which tool to call
response = model_with_tool.invoke(question)

# 3. Execute the requested tool
tool_call = response.tool_calls[0]

result = get_weather.invoke(tool_call["args"])

print("Tool result:", result)

# 4. Create the tool result message
tool_message = ToolMessage(
    content=result,
    tool_call_id = tool_call['id']
)
print('tool_messa', tool_message)

# 5. Send the conversation and tool result back to the model

final_response = model_with_tool.invoke([
    HumanMessage(content=question),
    response,
    tool_message
])

print('final_res', final_response.content)




Tool result: It's sunny in Boston
tool_messa content="It's sunny in Boston" tool_call_id='7a998f4f-d4bc-44c3-a1a9-b37a6e2444de'
final_res It's sunny in Boston. Would you like to know the weather forecast for the next 5 days?


### Multiple tool calls

In [8]:
@langchain_tool
def convert_celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit."""
    return (celsius * 9 / 5) + 32

# 5. Ask a question that needs both tools
question = """
Find the weather in Boston and convert 25 Celsius to Fahrenheit.
Use get_weather with location Boston and
convert_celsius_to_fahrenheit with celsius 25.
"""

# 6. Ask the model which tools to call
response = model_with_tool.invoke(question)

print("Generated tool calls:")
print(response.tool_calls)
print("\nNumber of tool calls:", len(response.tool_calls))

# 7. Create a lookup of available tools
tools_by_name = {
    "get_weather": get_weather,
    "convert_celsius_to_fahrenheit": convert_celsius_to_fahrenheit,
}

# 8. Execute every tool call generated by the model
tool_results = []

for tool_call in response.tool_calls:
    name = tool_call["name"]
    args = tool_call["args"]

    # Work around the schema-shaped args from this model
    if isinstance(args, dict) and "properties" in args:
        args = args["properties"]

    print(f"\nCalling tool: {name}")
    print("Arguments:", args)

    tool = tools_by_name.get(name)

    if tool is None:
        print("Unknown tool:", name)
        continue

    result = tool.invoke(args)
    tool_results.append((name, result))

    print("Result:", result)

# 9. Display all results
print("\nAll tool results:")
for name, result in tool_results:
    print(f"{name}: {result}")


Generated tool calls:
[{'name': 'get_weather', 'args': {'properties': {'location': 'Boston'}, 'type': 'object', 'required': ['location']}, 'id': 'e0fbfbae-3a76-42e4-a193-ef015156f5a3', 'type': 'tool_call'}]

Number of tool calls: 1

Calling tool: get_weather
Arguments: {'location': 'Boston'}
Result: It's sunny in Boston

All tool results:
get_weather: It's sunny in Boston


In [57]:
questions = [
    "Call get_weather with location Boston.",
    "Call convert_celsius_to_fahrenheit with celsius 25."
]

for question in questions:
    response = model_with_tool.invoke(question)

    print("\nQuestion:", question)
    print("Tool calls:", response.tool_calls)


Question: Call get_weather with location Boston.
Tool calls: [{'name': 'get_weather', 'args': {'required': ['location'], 'properties': {'location': 'Boston'}, 'type': 'object'}, 'id': 'dd1ea7d2-7402-4a6d-ad5e-84f7ed9782d2', 'type': 'tool_call'}]

Question: Call convert_celsius_to_fahrenheit with celsius 25.
Tool calls: [{'name': 'convert_celsius_to_fahrenheit', 'args': {'celsius': 25}, 'id': 'c9d8dfae-22b1-477c-9e09-9f0bc96e0792', 'type': 'tool_call'}]


In [9]:
tool_calls = [
    {
        "name": "get_weather",
        "args": {"location": "Boston"}
    },
    {
        "name": "convert_celsius_to_fahrenheit",
        "args": {"celsius": 25}
    }
]

tools_by_name = {
    "get_weather": get_weather,
    "convert_celsius_to_fahrenheit": convert_celsius_to_fahrenheit
}

for tool_call in tool_calls:
    tool = tools_by_name[tool_call["name"]]
    result = tool.invoke(tool_call["args"])

    print(tool_call["name"], "->", result)

get_weather -> It's sunny in Boston
convert_celsius_to_fahrenheit -> 77.0


In [41]:

from langchain_core.messages import HumanMessage, ToolMessage

questions = [
    "What is the weather in Boston?",
    "Also convert 25 Celsius to Fahrenheit."
]

# 1. Bind tools to the model
tools = [get_weather, convert_celsius_to_fahrenheit]
model_with_tools = model.bind_tools(tools)

# 2. Create a lookup for executing tools
tools_by_name = {tool.name: tool for tool in tools}

# 3. Store the complete conversation
messages = []
tool_results = []

for question in questions:
    # Send each question to the model
    response = model_with_tools.invoke(question)

    # Preserve the user message and AI response
    messages.extend([
        HumanMessage(content=question),
        response
    ])

    print('message', messages)

    # Process every tool call, not just the first one
    for tool_call in response.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]

        # Workaround for the malformed args from your local model
        if isinstance(args, dict) and "properties" in args:
            args = args["properties"]

        selected_tool = tools_by_name.get(name)

        if selected_tool is None:
            raise ValueError(f"Unknown tool: {name}")

        # Execute the tool
        result = selected_tool.invoke(args)
        tool_results.append({
            "tool": name,
            "result": result
        })

        # Associate the result with its exact tool call
        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

# 4. Display individual tool results
print("Tool results:")
for item in tool_results:
    print(f"{item['tool']}: {item['result']}")

# 5. Ask the model to combine the results into a final answer
messages.append(
    HumanMessage(
        content="Summarize the tool results and answer my original questions clearly."
    )
)

final_response = model.invoke(messages)

print("\nFinal answer:")
print(final_response.content)

message [HumanMessage(content='What is the weather in Boston?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-10-23T07:21:42.554229Z', 'done': True, 'done_reason': 'stop', 'total_duration': 401329083, 'load_duration': 7269541, 'prompt_eval_count': 204, 'prompt_eval_duration': 78448000, 'eval_count': 28, 'eval_duration': 299476000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a167a3-1004-7345-ba11-d54d1c905739-0', tool_calls=[{'name': 'get_weather', 'args': {'required': ['location'], 'properties': {'location': 'Boston'}, 'type': 'object'}, 'id': '81910be3-e85d-4625-8829-09f86c322e39', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 28, 'total_tokens': 232})]
message [HumanMessage(content='What is the weather in Boston?', additional_kwargs={}, response_metadata={}), AIMessage(content='', 

### Calling tool with agents

In [54]:
from langchain.agents import create_agent

tools = [
    get_weather,
    convert_celsius_to_fahrenheit,
]

agent = create_agent(
    model=model,
    tools=tools,
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Boston?"
        }
    ]
})


structure_model_response(result)




[
  {
    "content": "What is the weather in Boston?",
    "additional_kwargs": {},
    "response_metadata": {},
    "type": "human",
    "name": null,
    "id": "0f3efae0-ea28-4913-8f55-955a5728ad8f"
  },
  {
    "content": "",
    "additional_kwargs": {},
    "response_metadata": {
      "model": "llama3.2:1b",
      "created_at": "2026-10-23T10:19:06.147707Z",
      "done": true,
      "done_reason": "stop",
      "total_duration": 335495667,
      "load_duration": 691375,
      "prompt_eval_count": 204,
      "prompt_eval_duration": 22279000,
      "eval_count": 28,
      "eval_duration": 307384000,
      "logprobs": null,
      "model_name": "llama3.2:1b",
      "model_provider": "ollama"
    },
    "type": "ai",
    "name": null,
    "id": "lc_run--01a16845-78d3-76dd-8825-3a9f823ba015-0",
    "tool_calls": [
      {
        "name": "get_weather",
        "args": {
          "required": [
            "location"
          ],
          "properties": {
            "location": "Boston